## Likelihood analysis

In the annotation notebook, we have seen that we can add the likelihood as an annotation to each text signal. We can call the same function also to create a likelihood evaluation for each utterance in a sequence and the interations overall. This approach was proposed by:

```Mehri, S., & Eskenazi, M. (2020). USR: An unsupervised and reference free evaluation metric for dialog generation. arXiv preprint arXiv:2005.00456.```

They finetuned a RoBERTa model (USR) specifically to mimic human evaluation, as a reference-free approach. Reference-free means that is does not need a human-created response to compare the system response. Instead it determines the likelihood of the system response given the preceding conversation. The likelihood is calculated by taking the average likelihood of all tokens of the system response according to the model. The USR model was finetuned with conversations from TopicalChat and PersonaChat:

```Karthik Gopalakrishnan, Behnam Hedayatnia, Qinlang Chen, Anna Gottardi, Sanjeev Kwatra, Anu Venkatesh, Raefer Gabriel, Dilek Hakkani-Tur, and Amazon Alexa AI. 2019. Topical-chat: Towards knowledge-grounded open-domain conversations. Proc. Interspeech 2019, pages 1891–1895.```
```Saizheng Zhang, Emily Dinan, Jack Urbanek, Arthur Szlam, Douwe Kiela, and Jason Weston. 2018. Personalizing dialogue agents: I have a dog, do you have pets too? arXiv preprint arXiv:1801.07243.```

Their approach has moderate correlation with human judgments: .42 (TopicalChat) and .48 (PersonaChat). The USR model can be downloaded from the course drive: 

[usr-topicalchat-roberta_ft.zip](https://drive.google.com/file/d/1ODF-trnYeWm_hSkv9-D6-xHlnM47ToBy/view?usp=share_link)

Unpack the zip file and place it anywhere on your local machine. Adapt the path in this notebook below to your local copy of the model:

You can use any other BERT or RoBERTa model (ENCODER) from [hugggingface.co](https://huggingface.co) to score the likelihood. 


Each system token of the uterance under consideration is turned into a masked token to predict the most probable tokens according to the model (cut-off by ```len_top_tokens```). If the system token is in the list, it will receive the score from the model. If it is not in the list, the token scores ```0```.

## Prerequisites

This notebooks relies on the transformers package and the EMISSOR package for loading the EMISSOR scenarios. These packages can be installed through ```pip```:

In [1]:
# !pip install emissor
# !pip install numpy==1.26.4
# !pip install torch==2.2.2
# !pip install transformers==4.51.3

## Loading an ENCODER model for the masked-task from huggingface or disk

In [2]:
import re
import numpy
from transformers import pipeline, AutoTokenizer

model_name = "google-bert/bert-base-uncased"
#model_name = "FacebookAI/xlm-roberta-base"
#model_name = "FacebookAI/roberta-base"

# Path to your local copy of the USR model
model_name ="../../models/usr-topicalchat-roberta_ft"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = pipeline("fill-mask", model=model_name)

Device set to use mps:0


In [3]:
def mask_target_sentence(context, target):
    masked_targets = []
    ## We limit the length of the target as too long utterance break the token limit
    target_tokens = re.split(' ', target[:500])
    for index, token in enumerate(target_tokens):
        sequence = context + " "
        for token in target_tokens[:index]:
            sequence += token + " "
        sequence += tokenizer.mask_token
        for token in target_tokens[index + 1:]:
            sequence += " " + token
        masked_targets.append(sequence)
    return masked_targets, target_tokens

def sentence_likelihood(context, target):
    masked_targets, target_tokens = mask_target_sentence(context, target)
    expected_target = ""
    max_scores = []
    scores = []
    for masked_target, token in zip(masked_targets, target_tokens):
        results = model(masked_target)
        expected_target += results[0]['token_str'] + " "
        max_scores.append(results[0]['score'])
        match = False
        for result in results:
            if result['token_str'].lower().strip() == token.lower():
                scores.append(result['score'])
                match = True
                break
        if not match:
            scores.append(0)
    likelihood = sum(scores) / len(scores)
    max_likelihood = sum(max_scores) / len(max_scores)

    return likelihood, expected_target, max_likelihood

def score_pairs_for_likelihood(turns: []):
    for context, target in turns:
        llh, best_sentence, max_score = sentence_likelihood(context, target)
        print('Likelihood:', llh, 'Max score:', max_score, 'Best sentence:', best_sentence)

## Obtaining the conversations from EMISSOR

The next function gets the text signals from a conversation captured in an EMISSOR scenario.

In [7]:
import os
import pandas as pd
from emissor.persistence import ScenarioStorage
from emissor.representation.scenario import Modality
from emissor.representation.scenario import Signal, TextSignal
import emissor_util as util

In [9]:
EMISSOR="../emissor"
SCENARIO="b387db06-934e-405b-9d4e-f7e5c27b440a"

text_signals = util.get_text_signals_from_a_scenario(EMISSOR, SCENARIO)
llh_results = []
text = ""
context= ""
for index, text_signal in enumerate(text_signals):
    print(f"Processing turn {index}/{len(text_signals) - 1}")
    turn_id = text_signal.id
    speaker = util.get_speaker_from_text_signal(text_signal)
    text = text_signal.text
    llh, model_sentence, max_score = sentence_likelihood(context, text)
    row = {"Turn": turn_id, "Speaker": speaker, "Response": text, "llh": llh, "model_turn": model_sentence, "max_llh": max_score}
    llh_results.append(row)
    context = text

df = pd.DataFrame(llh_results)

evaluation_folder = os.path.join(EMISSOR, SCENARIO, 'evaluation')
if not os.path.exists(evaluation_folder):
    os.mkdir(evaluation_folder)
file_name =  SCENARIO+"_likelihood_evaluation.csv"
file = os.path.join(evaluation_folder, file_name)
df.to_csv(file, sep=";", index=False)

Processing turn 0/8
Processing turn 1/8
Processing turn 2/8
Processing turn 3/8
Processing turn 4/8
Processing turn 5/8
Processing turn 6/8
Processing turn 7/8
Processing turn 8/8


The conversation is saved to a CSV file in the ```evaluation``` subfolder in the scenario. It has the turns on each row with the turn identifier, the speaker and the reponse.
Furthermore, it has columsn for the likelihood score (llh), the token sequence with the highest probablity according to the model (model_turn) and the maximal likelihood the model would return for the response with the highest probability tokens (max_llh), which represents the upper celing for the model:

```
Turn	Speaker	Response	llh	model_turn	max_llh
fa29f41f-7bea-42b1-bc64-17c6a1bd320e	LEOLANI	Yo Do you want to talk to me Luis?	0.661853857	 .  do  you  want  to  talk  to  me  ? 	0.786302081
4df4c74e-44cf-471a-b0cb-5233d96dbb21	SPEAKER	Yes	0	 :) 	0.118416406
d9f4848a-87ab-4599-b6ef-e5bff1149e26	LEOLANI	I have nothing more to say.	0.560176643	 i  have  much  else  to  say 	0.676033412
f37db5d2-63b5-4a49-8a5f-8825c31aafc0	SPEAKER	I live in Amstelveen.	0.455679432	 i  am  in  dc 	0.670751117
```

Since we score both the user (SPEAKER) and the agent (LEOLANI), we can obtain the likelihood of each's responses and check if the user submitted more likely responses than the agent.

In [13]:
speaker_llh = []
agent_llh = []
speaker_max_llh = []
agent_max_llh = []
for llh in llh_results:
    print(llh["Response"], ":", llh["model_turn"])
    if llh["Speaker"]=="LEOLANI":
        agent_llh.append(llh["llh"])
        agent_max_llh.append(llh["max_llh"])
    else:
        speaker_llh.append(llh["llh"])
        speaker_max_llh.append(llh["max_llh"])

average_speaker_llh = sum(speaker_llh)/len(speaker_llh)
average_speaker_max_llh = sum(speaker_max_llh)/len(speaker_max_llh)
average_agent_llh = sum(agent_llh)/len(agent_llh)
average_agent_max_llh = sum(agent_max_llh)/len(agent_max_llh)

print('average_speaker_llh',average_speaker_llh)
print('average_speaker_max_llh',average_speaker_max_llh)
print('average_agent_llh',average_agent_llh)
print('average_agent_max_llh',average_agent_max_llh)

Yo Do you want to talk to me Luis? :  .  do  you  want  to  talk  to  me  ? 
Yes :  :) 
I have nothing more to say. :  i  have  much  else  to  say 
I live in Amstelveen. :  i  am  in  dc 
Has Luis visited the agent? : have  anyone  been  your  city 
I visited Piek. : he  think  him 
Do you know Piek? : do  you  like  about 
Ik wil het weten. Heeft Luis muzikale werken? : what  is  van  be e  is t  . 
Someone said that I know Piek. :  i  asked  ,  i  do  ! 
average_speaker_llh 0.24572213180363178
average_speaker_max_llh 0.3758374027286967
average_agent_llh 0.28168064396414494
average_agent_max_llh 0.4760598153641655


In this example, we see that the speaker has an average score of 0.25 and the agent a higher score of 0.28. What does that tell you?

## End of notebook